# The bead on a periodic sheet: the figures

The plots of `docs/bead_push_benchmark.md` section 9, made from the two run
matrices of `analysis/sticky_bead_runs.py`: the **sticky** bead (the full
harmonic bond, `beadPotential = harmonic`) and the bead that only **pushes**
(`beadPotential = wall`, no well), each under four reservoirs (box / global S0
/ local S0 / fixed) and four fluidities (frozen, nu = 0.05, 0.5, 5 per edge per
us). Every figure is drawn by `analysis/sticky_bead_analysis.py`; this notebook
loads each matrix once, writes each figure as **.svg and .png** into
`<matrix>/analysis/figures`, shows it, and ends with the two beads set against
each other. The animations (GIF) are optional, section 8.

The matrices are read from `SLIMED_STICKY_RUNS` and `SLIMED_PUSH_RUNS`, by
default `~/slimed_runs/sticky_bead/matrix` and `.../push_matrix`. A matrix
whose directory is missing is skipped. Nothing is simulated here.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
from IPython.display import Image, Markdown, display


def find_repo_root(start=None):
    """The SLIMED checkout this notebook sits in, wherever it is run from."""
    here = Path(start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / "CMakeLists.txt").is_file() and (d / "analysis").is_dir():
            return d
    raise RuntimeError("not inside a SLIMED checkout")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "analysis"))
import sticky_bead_analysis as sb

ROOTS = {
    "sticky": Path(os.environ.get("SLIMED_STICKY_RUNS", "~/slimed_runs/sticky_bead/matrix")).expanduser(),
    "push": Path(os.environ.get("SLIMED_PUSH_RUNS", "~/slimed_runs/sticky_bead/push_matrix")).expanduser(),
}
ROOTS = {k: v for k, v in ROOTS.items() if v.is_dir()}
mpl.rcParams["savefig.dpi"] = 220
for k, v in ROOTS.items():
    print(f"{k:7s} {v}")

---
## 0. The runs

Every directory under each root with a bead log. Loading reads the control
net, the surface points and the per-face energies of all 901 frames of each
run, about a minute per matrix of 13.

In [ ]:
def load_matrix(root):
    runs = []
    for d in sorted(p for p in root.iterdir() if (p / "inputbead.csv").exists()):
        try:
            runs.append(sb.StickyRun(str(d)))
            r = runs[-1]
            print(f"{d.name:18s} {'sticky' if r.sticky else 'pushing':8s} {r.n_frames} frames to {r.time[-1]:.0f} us, "
                  f"{r.n_free} free vertices, tile {r.L0.round(1)}")
        except Exception as exc:
            print(f"skipping {d.name}: {exc}")
    runs.sort(key=lambda r: (list(sb.RESERVOIR_STYLES).index(r.reservoir),
                             list(sb.FLUIDITY_COLORS).index(r.fluidity) if r.fluidity in sb.FLUIDITY_COLORS else 9))
    return runs


MATRIX = {k: load_matrix(v) for k, v in ROOTS.items()}
STATS = {k: [sb.summarize(r) for r in runs] for k, runs in MATRIX.items()}


class Matrix:
    def __init__(self, key):
        self.key, self.runs, self.stats = key, MATRIX[key], STATS[key]
        self.fig = ROOTS[key] / "analysis" / "figures"
        self.fig.mkdir(parents=True, exist_ok=True)
        self.t_end = min(r.time[-1] for r in self.runs)
        self.t_stop = min(self.runs[0].t_stop, self.t_end)
        self.vmax = max(np.percentile(r.bending_density(sb.frame_index(r, self.t_end)), 99) for r in self.runs)

    def figure(self, writer, name, *args, **kwargs):
        """Write one figure as .svg and .png (the writer builds it from the runs) and show the .png."""
        for ext in ("svg", "png"):
            writer(self.runs, str(self.fig / f"{name}.{ext}"), *args, **kwargs)
        display(Markdown(f"**{self.key} bead** — `{self.fig / name}.svg|png`"))
        display(Image(filename=str(self.fig / f"{name}.png")))


MATRICES = [Matrix(k) for k in MATRIX if MATRIX[k]]

---
## 1. Traces

Force on the bead, the vertices on it (bonded to the sticky bead, in contact
with the pushing one), tension and the reservoir's area; grey while the bead
moves, white while it holds.

In [ ]:
for m in MATRICES:
    #m.runs.remove(m.runs[4])
    m.figure(sb.write_traces, "traces")

## 2. Relaxation after the stop

Exponential fits of the force, the vertices on the bead and the tension over
the hold phase.

In [ ]:
for m in MATRICES:
    m.figure(sb.write_hold_relaxation, "hold_relaxation", m.stats)

## 3. Bending energy maps

Top view of the control net at the stop and at the end. Only the
energy-carrying faces are drawn, tiled by the period for the margin; the
mesh's image-band copies go stale under flips and are left out (see the
docs).

In [ ]:
for m in MATRICES:
    for t_us in sorted({m.t_stop, m.t_end}):
        m.figure(sb.energy_map_grid, f"energy_maps_{t_us:.0f}us", t_us, vmax=m.vmax)

## 4. Ring profiles

Mean bending energy density in rings around the bead's axis.

In [ ]:
for m in MATRICES:
    m.figure(sb.write_profiles, "energy_profiles", sorted({m.t_stop, m.t_end}))

## 5. Fourier analysis of the hold phase

The static imprint of the bead and the fluctuation spectrum of the sheet
outside its footprint, with the kc and sigma fits.

In [ ]:
for m in MATRICES:
    m.figure(sb.write_spectra, "spectra", m.stats)

## 6. The numbers

`stats.md` beside each matrix's figures; the tables of
`docs/bead_push_benchmark.md` section 9 are condensed from them.

In [ ]:
for m in MATRICES:
    sb.write_stats(m.stats, str(m.fig / "stats.md"), str(m.fig / "stats.csv"))
    display(Markdown(f"**{m.key} bead**\n\n" + (m.fig / "stats.md").read_text()))

## 7. Sticky against pushing

The same run of each matrix on one axis: colour by fluidity, solid for the
sticky bead, dashed for the one that only pushes. Written beside the sticky
matrix's figures, with the merged table.

In [ ]:
sticky = [r for m in MATRICES for r in m.runs if r.sticky]
push = [r for m in MATRICES for r in m.runs if not r.sticky]
if sticky and push:
    fig_dir = MATRICES[0].fig
    for ext in ("svg", "png"):
        sb.write_potential_comparison(sticky, push, str(fig_dir / f"potential_comparison.{ext}"))
    display(Image(filename=str(fig_dir / "potential_comparison.png")))
    both = sorted((s for m in MATRICES for s in m.stats), key=lambda s: (s["reservoir"], s["fluidity"], s["potential"]))
    sb.write_stats(both, str(fig_dir / "stats_comparison.md"), str(fig_dir / "stats_comparison.csv"))
    display(Markdown((fig_dir / "stats_comparison.md").read_text()))
else:
    print("one matrix only: nothing to compare")

## 8. Animations (optional)

One GIF per run and a four-panel comparison per matrix, about a minute each.
Set `MAKE_GIFS = True` to render them.

In [ ]:
MAKE_GIFS = False
COMPARE = ["box_frozen", "box_nu0.05", "box_nu0.5", "box_nu5"]
if MAKE_GIFS:
    for m in MATRICES:
        for r in m.runs:
            n = sb.write_animation([r], str(m.fig / f"anim_{r.name}.gif"), 150, 15, m.vmax)
            print(f"{m.key}: anim_{r.name}.gif: {n} frames")
        chosen = [r for r in m.runs if r.name in COMPARE][:4]
        if chosen:
            sb.write_animation(chosen, str(m.fig / "compare.gif"), 150, 15, m.vmax)
            print(f"{m.key}: compare.gif written")